In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p /content/Insurance-Fraud-Detection-Processed

In [ ]:
!unzip -q "/content/drive/MyDrive/Insurance-Fraud-Detection-Processed/train_balanced.zip" \
-d "/content/Insurance-Fraud-Detection-Processed/"

In [ ]:
!ls "/content/Insurance-Fraud-Detection-Processed"

Fraud  Non-Fraud


In [ ]:
!find "/content/Insurance-Fraud-Detection-Processed" -maxdepth 3 -type d

/content/Insurance-Fraud-Detection-Processed
/content/Insurance-Fraud-Detection-Processed/Fraud
/content/Insurance-Fraud-Detection-Processed/Non-Fraud


In [ ]:
!mkdir -p "/content/Insurance-Fraud-Detection-Processed/train_balanced"

!mv "/content/Insurance-Fraud-Detection-Processed/Fraud" \
"/content/Insurance-Fraud-Detection-Processed/train_balanced/"

!mv "/content/Insurance-Fraud-Detection-Processed/Non-Fraud" \
"/content/Insurance-Fraud-Detection-Processed/train_balanced/"

In [ ]:
!find "/content/Insurance-Fraud-Detection-Processed" -maxdepth 3 -type d

/content/Insurance-Fraud-Detection-Processed
/content/Insurance-Fraud-Detection-Processed/train_balanced
/content/Insurance-Fraud-Detection-Processed/train_balanced/Fraud
/content/Insurance-Fraud-Detection-Processed/train_balanced/Non-Fraud


In [ ]:
!mkdir -p "/content/Insurance-Fraud-Detection-Processed/val"

!unzip -q \
"/content/drive/MyDrive/Insurance-Fraud-Detection-Processed/val.zip" \
-d "/content/Insurance-Fraud-Detection-Processed/val"

In [ ]:
!find "/content/Insurance-Fraud-Detection-Processed/val" -maxdepth 2 -type d

/content/Insurance-Fraud-Detection-Processed/val
/content/Insurance-Fraud-Detection-Processed/val/Fraud
/content/Insurance-Fraud-Detection-Processed/val/Non-Fraud


In [ ]:
!mkdir -p "/content/Insurance-Fraud-Detection-Processed/test"

!unzip -q \
"/content/drive/MyDrive/Insurance-Fraud-Detection-Processed/test.zip" \
-d "/content/Insurance-Fraud-Detection-Processed/test"

In [ ]:
import os

BASE_DIR = "/content/Insurance-Fraud-Detection-Processed"

folders = {
    "Train Fraud": os.path.join(BASE_DIR, "train_balanced", "Fraud"),
    "Train Non-Fraud": os.path.join(BASE_DIR, "train_balanced", "Non-Fraud"),
    "Val Fraud": os.path.join(BASE_DIR, "val", "Fraud"),
    "Val Non-Fraud": os.path.join(BASE_DIR, "val", "Non-Fraud"),
    "Test Fraud": os.path.join(BASE_DIR, "test", "Fraud"),
    "Test Non-Fraud": os.path.join(BASE_DIR, "test", "Non-Fraud"),
}

for name, folder in folders.items():
    count = len([
        f for f in os.listdir(folder)
        if os.path.isfile(os.path.join(folder, f))
    ])
    print(f"{name}: {count}")

Train Fraud: 1360
Train Non-Fraud: 4250
Val Fraud: 30
Val Non-Fraud: 750
Test Fraud: 93
Test Non-Fraud: 1323


In [ ]:
!cp "/content/drive/MyDrive/Insurance-Fraud-Detection-Processed/class_weights.json" \
"/content/Insurance-Fraud-Detection-Processed/class_weights.json"

In [ ]:
import json

with open(
    "/content/Insurance-Fraud-Detection-Processed/class_weights.json",
    "r"
) as f:
    class_weights_named = json.load(f)

print(class_weights_named)

{'Fraud': 2.0625, 'Non-Fraud': 0.66}


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving step2_get_generators.py to step2_get_generators.py


In [ ]:
!ls

 drive				       step2_get_generators.py
 Insurance-Fraud-Detection-Processed  'step2_get_generators.py (1).py'
 sample_data


In [ ]:
from step2_get_generators import get_generators

In [ ]:
train_gen, val_gen, test_gen = get_generators()

Found 5610 images belonging to 2 classes.
Found 780 images belonging to 2 classes.
Found 1416 images belonging to 2 classes.

Class indices:
{'Fraud': 0, 'Non-Fraud': 1}


In [ ]:
import tensorflow as tf

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers
from tensorflow.keras import models

In [ ]:
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
base_model.trainable = False

In [ ]:
model = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dropout(0.3),

    layers.Dense(1, activation="sigmoid")
])

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,050,852 (15.45 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),

    loss="binary_crossentropy",

    metrics=["accuracy"]
)

In [ ]:
with open(
    "/content/Insurance-Fraud-Detection-Processed/class_weights.json",
    "r"
) as f:
    class_weights_named = json.load(f)

class_weights = {
    train_gen.class_indices["Fraud"]: class_weights_named["Fraud"],
    train_gen.class_indices["Non-Fraud"]: class_weights_named["Non-Fraud"]
}

print(class_weights)

{0: 2.0625, 1: 0.66}


In [ ]:
history = model.fit(
    train_gen,

    validation_data=val_gen,

    epochs=10,

    class_weight=class_weights
)

Epoch 1/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 181s 800ms/step - accuracy: 0.6168 - loss: 0.6416 - val_accuracy: 0.7897 - val_loss: 0.5665
Epoch 2/10
 15/176 ━━━━━━━━━━━━━━━━━━━━ 1:07 420ms/step - accuracy: 0.6669 - loss: 0.6055

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Reset test generator
test_gen.reset()

# Get predicted probabilities
y_prob = model.predict(test_gen, verbose=1).ravel()

# Convert probabilities to class predictions
y_pred = (y_prob >= 0.5).astype(int)

# True labels
y_true = test_gen.classes

# Check class mapping
print("Class indices:", test_gen.class_indices)

# Overall metrics
accuracy = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)

# Per-class metrics
precision = precision_score(
    y_true,
    y_pred,
    average=None,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    average=None,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    average=None,
    zero_division=0
)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

print("\n==============================")
print("FINAL TEST SET RESULTS")
print("==============================")

print(f"Accuracy : {accuracy:.4f}")
print(f"AUC      : {auc:.4f}")

for class_name, p, r, f in zip(
    test_gen.class_indices.keys(),
    precision,
    recall,
    f1
):
    print(f"\n{class_name}")
    print(f"Precision: {p:.4f}")
    print(f"Recall   : {r:.4f}")
    print(f"F1       : {f:.4f}")

print("\nConfusion Matrix:")
print(cm)

In [ ]:
model.save("/content/efficientnetb0.keras")

In [ ]:
import json

# Get the actual class names in the same order as class indices
class_names = list(test_gen.class_indices.keys())

# Create results dictionary
results = {
    "accuracy": float(accuracy),
    "auc": float(auc),

    "Fraud": {
        "precision": float(precision[test_gen.class_indices["Fraud"]]),
        "recall": float(recall[test_gen.class_indices["Fraud"]]),
        "f1": float(f1[test_gen.class_indices["Fraud"]])
    },

    "Non-Fraud": {
        "precision": float(precision[test_gen.class_indices["Non-Fraud"]]),
        "recall": float(recall[test_gen.class_indices["Non-Fraud"]]),
        "f1": float(f1[test_gen.class_indices["Non-Fraud"]])
    },

    "confusion_matrix": cm.tolist()
}

# Save JSON
with open("/content/results.json", "w") as f:
    json.dump(results, f, indent=4)

print("results.json saved successfully!")

In [ ]:
from google.colab import files

!cp /content/results.json /content/results.txt

files.download("/content/results.txt")